
# Crypto Ranked Asset Allocation — Momentum + Volatility + Correlation + Entropy

This notebook implements a crypto adaptation of a **Ranked Asset Allocation** framework using four equally weighted factors:

1. **Momentum** — rewards persistent positive price trends.
2. **Volatility** — penalizes unstable/high-risk assets.
3. **Correlation** — rewards assets that are less correlated with the rest of the crypto universe.
4. **Entropy** — penalizes noisy/unpredictable return behavior.

The default universe is five large crypto assets:

**BTC, ETH, BNB, SOL, XRP**

> This is a research/backtesting framework, not financial advice. The goal is to test whether the proposed factor logic adds value out-of-sample rather than assuming that it does.



## Strategy design

At each rebalance date, the model calculates a score for every asset:

\[
Score_i = w_M M_i + w_V V_i + w_C C_i + w_E E_i
\]

with the default:

\[
w_M=w_V=w_C=w_E=25\%
\]

Each factor is converted to a **cross-sectional percentile rank**, so the factors are comparable despite having different units.

### Factor direction

- Momentum: **higher is better**
- Volatility: **lower is better**
- Correlation: **lower average correlation is better**
- Entropy: **lower entropy is better**

The entropy measure uses the distribution of rolling return signs (+ / -) as a simple, interpretable proxy for directional randomness.

### Portfolio rule

Default:
- Rebalance monthly
- Rank all five assets
- Hold the **top 3**
- Equal-weight the selected assets
- Optional volatility scaling
- No look-ahead: signals are calculated using data available at the rebalance close and applied to the following period

All important parameters are exposed in one configuration cell so they can be optimized or stress-tested later.


In [ ]:

# Install if needed:
# %pip install yfinance pandas numpy matplotlib seaborn scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from scipy.stats import rankdata


In [ ]:

# =========================
# STRATEGY CONFIGURATION
# =========================

TICKERS = ["BTC-USD", "ETH-USD", "BNB-USD", "SOL-USD", "XRP-USD"]

START_DATE = "2020-01-01"
END_DATE = None                  # None = latest available data

LOOKBACK = 60                    # factor lookback in trading/calendar observations
MOMENTUM_LOOKBACK = 60
VOL_LOOKBACK = 30
CORR_LOOKBACK = 60
ENTROPY_LOOKBACK = 60

REBALANCE_FREQUENCY = "ME"       # Month End
TOP_N = 3                         # number of assets held
INITIAL_CAPITAL = 10_000

# Equal factor weights
WEIGHTS = {
    "momentum": 0.25,
    "volatility": 0.25,
    "correlation": 0.25,
    "entropy": 0.25,
}

# Optional risk controls
USE_VOL_TARGET = False
VOL_TARGET_ANNUAL = 0.40
MAX_ASSET_WEIGHT = 0.50

# Trading assumptions
TRANSACTION_COST = 0.001         # 10 bps per unit of turnover
SLIPPAGE = 0.0005                # 5 bps per unit of turnover

assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9
assert TOP_N <= len(TICKERS)

print("Universe:", TICKERS)
print("Top N:", TOP_N)
print("Factor weights:", WEIGHTS)


In [ ]:

# =========================
# DOWNLOAD PRICE DATA
# =========================

data = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False,
)

if isinstance(data.columns, pd.MultiIndex):
    close = data["Close"].copy()
else:
    close = data[["Close"]].copy()
    close.columns = TICKERS[:1]

close = close.dropna(how="all").ffill()

# Keep only assets with sufficient history
available = [c for c in TICKERS if c in close.columns and close[c].notna().sum() > LOOKBACK]
close = close[available].dropna(how="all")

returns = close.pct_change()

print("Downloaded:", close.index.min().date(), "to", close.index.max().date())
print("Assets:", list(close.columns))

display(close.tail())


In [ ]:

# =========================
# FACTOR FUNCTIONS
# =========================

def momentum_factor(price, lookback=MOMENTUM_LOOKBACK):
    """Simple price momentum: total return over lookback."""
    return price / price.shift(lookback) - 1


def volatility_factor(ret, lookback=VOL_LOOKBACK):
    """Annualized realized volatility. Lower is better."""
    return ret.rolling(lookback).std() * np.sqrt(365)


def correlation_factor(ret, lookback=CORR_LOOKBACK):
    """Average absolute correlation with the other assets. Lower is better."""
    output = pd.DataFrame(index=ret.index, columns=ret.columns, dtype=float)

    for date in ret.index:
        window = ret.loc[:date].tail(lookback)
        corr = window.corr()

        for asset in ret.columns:
            peers = corr[asset].drop(labels=[asset], errors="ignore").dropna()
            output.loc[date, asset] = peers.abs().mean() if len(peers) else np.nan

    return output


def binary_entropy(series, lookback=ENTROPY_LOOKBACK):
    """
    Shannon entropy of positive/negative daily returns.

    p = fraction of positive observations
    H = -p log2(p) - (1-p) log2(1-p)

    H is normalized to [0, 1].
    Lower entropy = more directional persistence.
    """
    sign = (series > 0).astype(float)

    p = sign.rolling(lookback).mean()
    eps = 1e-12

    h = -(p * np.log2(p.clip(eps, 1-eps))
          + (1-p) * np.log2((1-p).clip(eps, 1-eps)))

    return h


def entropy_factor(ret, lookback=ENTROPY_LOOKBACK):
    return pd.DataFrame({
        c: binary_entropy(ret[c], lookback)
        for c in ret.columns
    })


def cross_sectional_percentile(df, higher_is_better=True):
    """Percentile rank each date across assets."""
    ranks = df.rank(axis=1, pct=True, method="average")
    return ranks if higher_is_better else 1 - ranks


In [ ]:

# =========================
# CALCULATE RAW FACTORS
# =========================

momentum = pd.DataFrame({
    c: momentum_factor(close[c])
    for c in close.columns
})

volatility = pd.DataFrame({
    c: volatility_factor(returns[c])
    for c in close.columns
})

correlation = correlation_factor(returns)

entropy = entropy_factor(returns)

display(momentum.tail())


In [ ]:

# =========================
# NORMALIZE FACTORS
# =========================

momentum_score = cross_sectional_percentile(momentum, higher_is_better=True)
volatility_score = cross_sectional_percentile(volatility, higher_is_better=False)
correlation_score = cross_sectional_percentile(correlation, higher_is_better=False)
entropy_score = cross_sectional_percentile(entropy, higher_is_better=False)

factor_scores = {
    "momentum": momentum_score,
    "volatility": volatility_score,
    "correlation": correlation_score,
    "entropy": entropy_score,
}

composite_score = (
    WEIGHTS["momentum"] * momentum_score
    + WEIGHTS["volatility"] * volatility_score
    + WEIGHTS["correlation"] * correlation_score
    + WEIGHTS["entropy"] * entropy_score
)

display(composite_score.tail())


In [ ]:

# =========================
# RANKING TABLE
# =========================

def ranking_snapshot(date):
    rows = pd.DataFrame({
        "Momentum": momentum.loc[date],
        "Volatility": volatility.loc[date],
        "Avg Abs Correlation": correlation.loc[date],
        "Entropy": entropy.loc[date],
        "Momentum Score": momentum_score.loc[date],
        "Volatility Score": volatility_score.loc[date],
        "Correlation Score": correlation_score.loc[date],
        "Entropy Score": entropy_score.loc[date],
        "Composite Score": composite_score.loc[date],
    })

    return rows.sort_values("Composite Score", ascending=False)

latest_valid_date = composite_score.dropna(how="all").index[-1]

ranking_snapshot(latest_valid_date).round(4)



## Backtest engine

The backtest uses **month-end rebalancing**.

Important implementation detail: the signal at month-end is shifted into the next period. This prevents using the current month's realized return to decide the portfolio that supposedly existed during that same month.

Transaction costs are charged from portfolio turnover.


In [ ]:

# =========================
# PORTFOLIO WEIGHTS
# =========================

rebalance_dates = composite_score.resample(REBALANCE_FREQUENCY).last().index
rebalance_dates = [d for d in rebalance_dates if d in composite_score.index]

weights = pd.DataFrame(0.0, index=close.index, columns=close.columns)

for d in rebalance_dates:
    score = composite_score.loc[d].dropna()

    if len(score) == 0:
        continue

    selected = score.nlargest(min(TOP_N, len(score))).index.tolist()

    w = pd.Series(0.0, index=close.columns)
    w[selected] = 1.0 / len(selected)

    # Optional cap
    w = w.clip(upper=MAX_ASSET_WEIGHT)

    # Re-normalize
    if w.sum() > 0:
        w = w / w.sum()

    # Apply from the following observation onward
    next_dates = close.index[close.index > d]
    if len(next_dates):
        weights.loc[next_dates[0]:, :] = w.values

weights = weights.replace(0, np.nan).ffill().fillna(0)

display(weights.tail())


In [ ]:

# =========================
# BACKTEST
# =========================

portfolio_gross_returns = (weights.shift(1) * returns).sum(axis=1)

turnover = weights.diff().abs().sum(axis=1).fillna(0)
cost_rate = TRANSACTION_COST + SLIPPAGE
costs = turnover * cost_rate

portfolio_returns = portfolio_gross_returns - costs
equity = INITIAL_CAPITAL * (1 + portfolio_returns.fillna(0)).cumprod()

# BTC buy-and-hold benchmark
btc = close["BTC-USD"].pct_change().fillna(0)
btc_equity = INITIAL_CAPITAL * (1 + btc).cumprod()

results = pd.DataFrame({
    "Strategy": portfolio_returns,
    "BTC": btc,
})

results.tail()


In [ ]:

# =========================
# PERFORMANCE METRICS
# =========================

def performance_metrics(ret, equity_curve, periods_per_year=365):
    ret = ret.dropna()
    equity_curve = equity_curve.dropna()

    total_return = equity_curve.iloc[-1] / equity_curve.iloc[0] - 1
    years = len(ret) / periods_per_year

    cagr = (equity_curve.iloc[-1] / equity_curve.iloc[0]) ** (1 / years) - 1 if years > 0 else np.nan

    ann_vol = ret.std() * np.sqrt(periods_per_year)
    sharpe = ret.mean() / ret.std() * np.sqrt(periods_per_year) if ret.std() > 0 else np.nan

    downside = ret[ret < 0].std()
    sortino = ret.mean() / downside * np.sqrt(periods_per_year) if downside > 0 else np.nan

    drawdown = equity_curve / equity_curve.cummax() - 1
    max_dd = drawdown.min()

    calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

    win_rate = (ret > 0).mean()

    return pd.Series({
        "Total Return": total_return,
        "CAGR": cagr,
        "Annualized Volatility": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": max_dd,
        "Calmar": calmar,
        "Daily Win Rate": win_rate,
    })


strategy_metrics = performance_metrics(portfolio_returns, equity)
btc_metrics = performance_metrics(btc, btc_equity)

metrics = pd.DataFrame({
    "Ranked Crypto Strategy": strategy_metrics,
    "BTC Buy & Hold": btc_metrics,
})

display(metrics.round(4))


In [ ]:

# =========================
# EQUITY CURVES
# =========================

plt.figure(figsize=(13, 6))
plt.plot(equity.index, equity, label="Ranked Crypto Strategy")
plt.plot(btc_equity.index, btc_equity, label="BTC Buy & Hold")
plt.yscale("log")
plt.title("Equity Curve — Log Scale")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:

# =========================
# DRAWDOWN
# =========================

strategy_dd = equity / equity.cummax() - 1
btc_dd = btc_equity / btc_equity.cummax() - 1

plt.figure(figsize=(13, 5))
plt.plot(strategy_dd.index, strategy_dd, label="Strategy")
plt.plot(btc_dd.index, btc_dd, label="BTC")
plt.title("Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:

# =========================
# FACTOR CONTRIBUTION VIEW
# =========================

latest = pd.DataFrame({
    "Momentum": momentum_score.loc[latest_valid_date],
    "Volatility": volatility_score.loc[latest_valid_date],
    "Correlation": correlation_score.loc[latest_valid_date],
    "Entropy": entropy_score.loc[latest_valid_date],
})

latest["Composite"] = composite_score.loc[latest_valid_date]

latest.sort_values("Composite", ascending=False).round(3)



## Walk-forward research ideas

Do **not** optimize all parameters on the full sample and then trust the resulting backtest.

Recommended research sequence:

### 1. Momentum robustness
Test:
- 20 / 30 / 60 / 90 / 120-day momentum
- simple return momentum
- log-price momentum
- moving-average distance
- breakout momentum
- volatility-adjusted momentum

### 2. Entropy alternatives
The current entropy measure is intentionally simple. Test:
- return-sign entropy
- Shannon entropy of return bins
- permutation entropy
- approximate entropy
- sample entropy

The key research question is:

> Does entropy add predictive information after momentum and volatility are already included?

### 3. Correlation alternatives
Test:
- average pairwise correlation
- average absolute correlation
- correlation to BTC
- correlation to the equal-weight crypto index
- downside correlation
- exponentially weighted correlation

### 4. Portfolio construction
Compare:
- Top 1
- Top 2
- Top 3
- Top 4
- all 5
- score-weighted allocation
- inverse-volatility allocation
- volatility-targeted allocation

### 5. Factor-weight experiments

Start with:

\[
25/25/25/25
\]

Then test controlled variations such as:

- 40/20/20/20
- 20/30/30/20
- 30/20/30/20
- 30/20/20/30

Use **walk-forward validation** rather than selecting the best combination from the entire historical sample.


In [ ]:

# =========================
# SIMPLE WEIGHT SENSITIVITY TEST
# =========================

weight_sets = {
    "Equal 25/25/25/25": (0.25, 0.25, 0.25, 0.25),
    "Momentum Heavy": (0.40, 0.20, 0.20, 0.20),
    "Risk/Correlation Heavy": (0.20, 0.30, 0.30, 0.20),
    "Entropy Heavy": (0.20, 0.20, 0.20, 0.40),
}

def build_composite(weight_tuple):
    wm, wv, wc, we = weight_tuple
    return wm * momentum_score + wv * volatility_score + wc * correlation_score + we * entropy_score

sensitivity = {}

for name, wt in weight_sets.items():
    score = build_composite(wt)
    wtest = pd.DataFrame(0.0, index=close.index, columns=close.columns)

    dates = score.resample(REBALANCE_FREQUENCY).last().index

    for d in dates:
        if d not in score.index:
            continue

        s = score.loc[d].dropna()
        selected = s.nlargest(min(TOP_N, len(s))).index

        ww = pd.Series(0.0, index=close.columns)
        ww[selected] = 1 / len(selected)

        future = close.index[close.index > d]
        if len(future):
            wtest.loc[future[0]:, :] = ww.values

    wtest = wtest.ffill().fillna(0)

    r = (wtest.shift(1) * returns).sum(axis=1)
    t = wtest.diff().abs().sum(axis=1).fillna(0)
    r = r - t * cost_rate

    eq = INITIAL_CAPITAL * (1 + r.fillna(0)).cumprod()
    sensitivity[name] = performance_metrics(r, eq)

sensitivity_df = pd.DataFrame(sensitivity)
display(sensitivity_df.round(4))



## Interpretation checklist

A good result is **not simply the strategy with the highest CAGR**.

Look for:

- Higher Sharpe and Sortino
- Lower maximum drawdown
- Reasonable turnover
- Stability across different lookbacks
- Stability across different crypto universes
- Performance after transaction costs
- Similar behavior in bull, bear and sideways regimes
- No single token dominating the entire result
- No single parameter combination producing the majority of returns

### Most important experiment

Run an ablation study:

1. Momentum only
2. Momentum + volatility
3. Momentum + volatility + correlation
4. Momentum + volatility + correlation + entropy

If adding entropy does **not** improve out-of-sample performance or drawdown characteristics, there may be little reason to keep it.

That directly tests the core hypothesis rather than assuming entropy must be useful.


In [ ]:

# =========================
# FACTOR ABLATION STUDY
# =========================

def run_ablation(active_factors, top_n=TOP_N):
    score = pd.DataFrame(0.0, index=close.index, columns=close.columns)

    active_weight = 1 / len(active_factors)

    for f in active_factors:
        score += active_weight * factor_scores[f]

    wtest = pd.DataFrame(0.0, index=close.index, columns=close.columns)

    for d in score.resample(REBALANCE_FREQUENCY).last().index:
        if d not in score.index:
            continue

        s = score.loc[d].dropna()
        if len(s) == 0:
            continue

        selected = s.nlargest(min(top_n, len(s))).index

        ww = pd.Series(0.0, index=close.columns)
        ww[selected] = 1 / len(selected)

        future = close.index[close.index > d]
        if len(future):
            wtest.loc[future[0]:, :] = ww.values

    wtest = wtest.ffill().fillna(0)

    r = (wtest.shift(1) * returns).sum(axis=1)
    t = wtest.diff().abs().sum(axis=1).fillna(0)
    r = r - t * cost_rate

    eq = INITIAL_CAPITAL * (1 + r.fillna(0)).cumprod()

    return performance_metrics(r, eq)


ablations = {
    "Momentum": ["momentum"],
    "Momentum + Vol": ["momentum", "volatility"],
    "Momentum + Vol + Corr": ["momentum", "volatility", "correlation"],
    "All 4 Factors": ["momentum", "volatility", "correlation", "entropy"],
}

ablation_df = pd.DataFrame({
    name: run_ablation(factors)
    for name, factors in ablations.items()
})

display(ablation_df.round(4))



## Next step: make this a production research model

A stronger version of this notebook should add:

- Dynamic top-5 universe selection by market cap
- Binance/Coinbase data instead of Yahoo Finance
- Weekly and daily rebalance comparison
- Walk-forward optimization
- Purged time-series cross-validation
- Regime detection
- Volatility targeting
- Stablecoin/cash allocation when all scores are weak
- Maximum drawdown controls
- Bootstrap confidence intervals
- Monte Carlo trade/order reshuffling
- Out-of-sample test period
- Parameter heatmaps
- Factor IC / rank-IC analysis
- Turnover decomposition
- Tax/fee/slippage sensitivity
- Live signal generation and webhook output

The most valuable research question is not **"Can I make the backtest profitable?"**

It is:

> **Does each additional factor provide incremental, statistically robust information after controlling for the factors already in the model?**
